# Etapa 2: Experimentos de Pre-Processamento

Requisitos obrigatorios abordados neste notebook:
1. Tratamento de dados redundantes/inconsistentes.
2. Comparacao de 2 tecnicas de balanceamento (ex.: SMOTE vs. RandomUnderSampler).
3. Codificacao simbolico-numerica justificada (efeito da alta cardinalidade em service: OneHot vs. Frequency Encoding).
4. Normalizacao/transformacao de atributos numericos (StandardScaler vs. RobustScaler).
5. Reducao de dimensionalidade ou selecao de atributos (PCA vs. SelectKBest).

REGRA METODOLOGICA CONTRA DATA LEAKAGE:
Todas as tecnicas sao ajustadas estritamente usando o conjunto de treino via Pipeline.

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
from src.data_loader import prepare_dataset
from src.preprocessing import build_preprocessor
from src.balancing import get_sampler
from src.models import get_classifier, build_pipeline
from src.evaluation import evaluate_cross_validation

# Carrega X_train e y_train com difficulty_level e num_outbound_cmds ja removidos
X_train, y_train = prepare_dataset("train")
print("X_train shape:", X_train.shape)
print("y_train contagem por classe:")
print(y_train.value_counts())

## 1. Definicao das Estrategias de Pre-Processamento

Aqui configuramos os blocos de transformacao que serao comparados:

In [ ]:
# Pre-processador A: One-Hot basico + StandardScaler
prep_onehot_std = build_preprocessor(categorical_strategy="onehot", scaler_type="standard")

# Pre-processador B: Frequency Encoding para service + RobustScaler
prep_freq_robust = build_preprocessor(categorical_strategy="frequency_service", scaler_type="robust")

# Pre-processador C: Com selecao de atributos (SelectKBest)
prep_kbest = build_preprocessor(categorical_strategy="frequency_service", scaler_type="robust", feature_reduction="select_k_best", n_features_or_components=25)

print("Pipelines de pre-processamento instanciados.")

## 2. Definicao dos Samplers de Balanceamento

Testamos SMOTE vs. RandomUnderSampler vs. Baseline (sem balanceamento).

In [ ]:
sampler_none = get_sampler("none")
sampler_smote = get_sampler("smote", random_state=42)
sampler_under = get_sampler("random_undersampler", random_state=42)

print("Samplers configurados:", ["None", "SMOTE", "RandomUnderSampler"])

## 3. Modelo Base para Comparar Pre-Processamento

Utilizamos um classificador rapido (ex: Decision Tree ou subconjunto de treino) para avaliar o impacto relativo de cada combinacao via StratifiedKFold.

In [ ]:
# Exemplo de execucao de teste rapido
clf_dt = get_classifier("decision_tree")
pipeline_teste = build_pipeline(prep_freq_robust, sampler_smote, clf_dt)

# Para testes rapidos de validacao cruzada, e possivel usar uma fracao estratificada do treino
print(pipeline_teste)